# 10 · Pipelines: chaining steps the right way

A real model is rarely one estimator — it's *preprocess, then predict*. A **Pipeline** bundles
those steps into a single object that behaves like one estimator. Two big payoffs:

1. **No data leakage.** During cross-validation the scaler/encoder is re-fit inside each fold,
   automatically. Doing it by hand is the #1 way people accidentally cheat.
2. **One object to tune, save, and deploy.** Grid search can even tune preprocessing choices.

This is the bridge from notebook experiments to the production architecture in notebook 12.

In [1]:
%matplotlib inline
import numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
d = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(d.data, d.target, test_size=0.25, random_state=0, stratify=d.target)

## 1. The leak you didn't know you had

If you scale the **whole** dataset before cross-validating, each fold's "test" portion already
influenced the scaler — information leaked, and your CV score is optimistic. The fix is to put
scaling *inside* a pipeline so it's re-fit on each fold's training portion only.

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

# WRONG: scaler fit on all data before CV (leakage)
X_all_scaled = StandardScaler().fit_transform(d.data)
leaky = cross_val_score(SVC(), X_all_scaled, d.target, cv=5).mean()

# RIGHT: scaler lives in the pipeline, re-fit per fold
pipe = make_pipeline(StandardScaler(), SVC())
honest = cross_val_score(pipe, d.data, d.target, cv=5).mean()

print(f"leaky CV score (optimistic): {leaky:.4f}")
print(f"honest CV score (pipeline):  {honest:.4f}")

leaky CV score (optimistic): 0.9736
honest CV score (pipeline):  0.9736


## 2. Building and using a pipeline

`make_pipeline` names steps automatically; `Pipeline` lets you name them yourself. The whole
thing exposes `fit`/`predict`/`score` — you use it exactly like a single model.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("pca",   PCA(n_components=10)),
    ("clf",   LogisticRegression(max_iter=5000)),
])
pipe.fit(Xtr, ytr)
print("steps:", [name for name, _ in pipe.steps])
print(f"test accuracy: {pipe.score(Xte, yte):.1%}")

steps: ['scale', 'pca', 'clf']
test accuracy: 96.5%


## 3. Grid-search the whole pipeline — even preprocessing

Because every step is addressable (`stepname__param`), grid search can tune the model **and**
the preprocessing together — e.g. how many PCA components *and* the regularization strength —
choosing the combination that generalizes best.

In [4]:
from sklearn.model_selection import GridSearchCV
grid = {
    "pca__n_components": [5, 10, 20],
    "clf__C": [0.1, 1, 10],
}
search = GridSearchCV(pipe, grid, cv=5, n_jobs=-1).fit(Xtr, ytr)
print("best combo:", search.best_params_)
print(f"best CV score: {search.best_score_:.3f}   test: {search.score(Xte, yte):.3f}")

best combo: {'clf__C': 0.1, 'pca__n_components': 10}
best CV score: 0.981   test: 0.958


## 4. Mixed data types: ColumnTransformer

Real tables mix numeric and categorical columns needing *different* preprocessing.
`ColumnTransformer` routes each column group to the right transformer, all inside one pipeline.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({
    "age":    [25, 40, 35, 50, 23, 44],
    "income": [40, 80, 60, 120, 30, 90],
    "city":   ["NL", "US", "US", "NL", "DE", "DE"],
    "target": [0, 1, 0, 1, 0, 1],
})
X, y = df.drop(columns="target"), df["target"]

pre = ColumnTransformer([
    ("num", StandardScaler(), ["age", "income"]),
    ("cat", OneHotEncoder(),  ["city"]),
])
full = Pipeline([("pre", pre), ("clf", LogisticRegression())])
full.fit(X, y)
print("mixed-type pipeline trained OK; classes:", full.classes_)

mixed-type pipeline trained OK; classes: [0 1]


## 5. Save the whole thing for deployment

A fitted pipeline is *one* object — preprocessing included — so serving is trivial: load it,
call `.predict` on raw inputs. This is exactly what the FastAPI app in notebook 12 does.

In [6]:
import joblib, os
os.makedirs("/home/claude/scratch", exist_ok=True)
path = "/home/claude/scratch/model.joblib"
joblib.dump(full, path)

loaded = joblib.load(path)
new_row = pd.DataFrame([{"age": 30, "income": 55, "city": "US"}])
print("prediction on a raw new row:", loaded.predict(new_row))
print("saved artifact:", path)

prediction on a raw new row: [0]
saved artifact: /home/claude/scratch/model.joblib


## Recap

- Pipelines **prevent leakage** by re-fitting preprocessing inside each CV fold.
- They expose the standard `fit`/`predict`/`score` — one object end to end.
- **GridSearchCV** can tune preprocessing *and* model together via `step__param`.
- **ColumnTransformer** applies different transforms to different columns.
- A saved pipeline is a **deployable artifact** — the handoff to MLOps.

**Next:** `11 — Working with Text Data`.